In [1]:
import pickle
import pandas as pd
import numpy as np

# Load model and scaler
with open('models/credit_default_model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('models/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

print(f"Loaded Model: {type(model).__name__}")
print(f"Expected number of features: {len(scaler.feature_names_in_)}")

Loaded Model: RandomForestClassifier
Expected number of features: 28


In [2]:
# 1. Load the original dataset
df = pd.read_csv('UCI_Credit_Card.csv')

# 2. Rename columns
df = df.rename(columns={'PAY_0': 'PAY_1', 'default.payment.next.month': 'default'})
if 'ID' in df.columns:
    df.drop('ID', axis=1, inplace=True)

# 3. Clean categorical anomalies
df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
df['MARRIAGE'] = df['MARRIAGE'].replace({0: 3})

# 4. Apply the same Feature Engineering
df['UTILIZATION'] = df['BILL_AMT1'] / (df['LIMIT_BAL'] + 1e-5)
bill_cols = ['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']
pay_cols = ['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
df['TOTAL_BILL'] = df[bill_cols].sum(axis=1)
df['TOTAL_PAY'] = df[pay_cols].sum(axis=1)
df['PAY_RATIO'] = df['TOTAL_PAY'] / (df['TOTAL_BILL'].clip(lower=1))
pay_status_cols = ['PAY_1', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
df['DELAY_COUNT'] = (df[pay_status_cols] > 0).sum(axis=1)

# 5. Extract a sample of 5 customers
sample_X = df.drop('default', axis=1).head(5)
sample_y = df['default'].head(5)

# Predict
predictions = model.predict(sample_X)
probabilities = model.predict_proba(sample_X)[:, 1]

# Display comparison
results = pd.DataFrame({
    'Actual': sample_y.values,
    'Predicted': predictions,
    'Default Probability (%)': (probabilities * 100).round(2)
})
display(results)


,Actual,Predicted,Default Probability (%)
0,1,1,74.70
1,1,0,41.91
2,0,0,10.05
3,0,0,14.97
4,0,0,8.53


In [6]:
def predict_new_customer(customer_dict):
    # Convert to DataFrame
    customer_df = pd.DataFrame([customer_dict])
    
    # Calculate engineered features automatically
    customer_df['UTILIZATION'] = customer_df['BILL_AMT1'] / (customer_df['LIMIT_BAL'] + 1e-5)
    
    bill_cols = ['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']
    pay_cols = ['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
    customer_df['TOTAL_BILL'] = customer_df[bill_cols].sum(axis=1)
    customer_df['TOTAL_PAY'] = customer_df[pay_cols].sum(axis=1)
    customer_df['PAY_RATIO'] = customer_df['TOTAL_PAY'] / (customer_df['TOTAL_BILL'].clip(lower=1))
    
    pay_status_cols = ['PAY_1', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
    customer_df['DELAY_COUNT'] = (customer_df[pay_status_cols] > 0).sum(axis=1)
    
    # Ensure columns match exact training order
    customer_df = customer_df[scaler.feature_names_in_]
    
    # Make prediction
    prediction = model.predict(customer_df)[0]
    probability = model.predict_proba(customer_df)[0][1]
    
    status = "HIGH RISK (Likely to Default)" if prediction == 1 else "LOW RISK (Likely to Pay)"
    
    print(f"Result: {status}")
    print(f"Default Probability: {probability:.1%}")
    return prediction, probability


In [7]:
# Example 1: High-risk customer (maxed out limit, 2 months delayed payments)
high_risk_client = {
    'LIMIT_BAL': 20000, 'SEX': 1, 'EDUCATION': 2, 'MARRIAGE': 1, 'AGE': 25,
    'PAY_1': 2, 'PAY_2': 2, 'PAY_3': 0, 'PAY_4': 0, 'PAY_5': 0, 'PAY_6': 0,
    'BILL_AMT1': 19500, 'BILL_AMT2': 19000, 'BILL_AMT3': 18000, 'BILL_AMT4': 17000, 'BILL_AMT5': 16000, 'BILL_AMT6': 15000,
    'PAY_AMT1': 0, 'PAY_AMT2': 0, 'PAY_AMT3': 1000, 'PAY_AMT4': 1000, 'PAY_AMT5': 1000, 'PAY_AMT6': 1000
}

predict_new_customer(high_risk_client)


Result: HIGH RISK (Likely to Default)
Default Probability: 59.8%


(np.int64(1), np.float64(0.5984551852387837))

In [8]:
# Example 2: Low-risk customer (high credit limit, pays on time)
low_risk_client = {
    'LIMIT_BAL': 250000, 'SEX': 2, 'EDUCATION': 1, 'MARRIAGE': 2, 'AGE': 35,
    'PAY_1': -1, 'PAY_2': -1, 'PAY_3': -1, 'PAY_4': -1, 'PAY_5': -1, 'PAY_6': -1,
    'BILL_AMT1': 5000, 'BILL_AMT2': 4000, 'BILL_AMT3': 3000, 'BILL_AMT4': 2000, 'BILL_AMT5': 1000, 'BILL_AMT6': 500,
    'PAY_AMT1': 5000, 'PAY_AMT2': 4000, 'PAY_AMT3': 3000, 'PAY_AMT4': 2000, 'PAY_AMT5': 1000, 'PAY_AMT6': 500
}

predict_new_customer(low_risk_client)


Result: LOW RISK (Likely to Pay)
Default Probability: 9.8%


(np.int64(0), np.float64(0.09779055114845063))